In [2]:
import numpy as np
import pandas as pd
import os, sys, glob
%load_ext autoreload
%autoreload 2
import camcan_utils as utils
import nibabel as nib
from config import *
from nibabel import Nifti1Image
from scipy import stats
from nilearn.maskers import NiftiMasker, NiftiLabelsMasker
import matplotlib.pyplot as plt
import scprep, tphate, phate
from nilearn import plotting
import shutil
import seaborn as sns
from brainiak.searchlight.searchlight import Searchlight
from camcan_utils import get_brain_cmap
import skdim
import ide_helpers as ide

/gpfs/milgram/project/casey/elb77/conda_envs/env_tda/lib/python3.7/site-packages/nilearn/__init__.py:69: FutureWarning: Python 3.7 support is deprecated and will be removed in release 0.12 of Nilearn. Consider switching to Python 3.9 or 3.10.
  _python_deprecation_warnings()


In [5]:
mask = utils.get_intersect_mask(35)
cmap = utils.get_brain_cmap()
# aggregate results
coords = np.where(mask.get_fdata() == 1)
# start with ISC
for t in ['rest','movie','SMT']:
    fns = sorted(glob.glob(f'{CAMCAN_RESULTS_DIR}/ISC/LOSO/*filter_35_{t}*ISC*.nii.gz'))
    print(f'found {len(fns)} for {t} ISC')
    
    # plot the group average
    temp = np.zeros(mask.shape)
    results_vecs = []
    for i, f in enumerate(fns):
        d = nib.load(f).get_fdata()
        temp = np.add(temp, d)
        vec = d[coords[0][:], coords[1][:], coords[2][:]]
        results_vecs.append(vec)
    temp /= len(fns)
    r = np.array(results_vecs)
    print(r.shape)
    np.save(f'{CAMCAN_RESULTS_DIR}/ISC/{t.upper()}_ISC_all_subjects_results.npy', r)
    
    
    n = Nifti1Image(temp, affine=mask.affine)
    nib.save(n, f'{CAMCAN_RESULTS_DIR}/ISC/group_average_{t}_ISC_whole_brain_SL_rad5.nii.gz')
    plot_fn = f'CamCAN/plots/group_average_{t}_ISC_whole_brain_SL_rad5.png'
    plotting.plot_img_on_surf(n, 
                              views=['lateral','medial'], 
                              cmap=cmap, 
                              inflate=True, 
                              bg_on_data=False, 
                                      alpha=0.8,
                              output_file=plot_fn,
                              darkness=0.8, 
                              threshold=0.1,
                              vmax=0.6, title=f'Camcan {t} ISC',
                             )
    plt.close()
    print(plot_fn)
    

found 117 for rest ISC
(117, 86741)
CamCAN/plots/group_average_rest_ISC_whole_brain_SL_rad5.png
found 117 for movie ISC
(117, 86741)
CamCAN/plots/group_average_movie_ISC_whole_brain_SL_rad5.png
found 117 for SMT ISC
(117, 86741)
CamCAN/plots/group_average_SMT_ISC_whole_brain_SL_rad5.png


In [9]:
for t in ['rest','movie','SMT']:
    for tp in ['optt','autocorr']:
        
        fns = sorted(glob.glob(f'{CAMCAN_RESULTS_DIR}/TPHATE_optt/LOSO/*{t}*{tp}*.nii.gz'))
        print(f'found {len(fns)} for {t} {tp}')

        # plot the group average
        temp = np.zeros(mask.shape)
        results_vecs = []
        for i, f in enumerate(fns):
            d = nib.load(f).get_fdata()
            temp = np.add(temp, d)
            vec = d[coords[0][:], coords[1][:], coords[2][:]]
            results_vecs.append(vec)
        temp /= len(fns)
        r = np.array(results_vecs)
        print(r.shape)
        np.save(f'{CAMCAN_RESULTS_DIR}/TPHATE_optt/{t.upper()}_tphate_{tp}_all_subjects_results.npy', r)


        n = Nifti1Image(temp, affine=mask.affine)
        nib.save(n, f'{CAMCAN_RESULTS_DIR}/TPHATE_optt/group_average_{t}_tphate_{tp}_whole_brain_SL_rad5.nii.gz')
        plot_fn = f'CamCAN/plots/group_average_{t}_tphate_{tp}_whole_brain_SL_rad5.png'
        plotting.plot_img_on_surf(n, 
                                  views=['lateral','medial'], 
                                  cmap=cmap, 
                                  inflate=True, 
                                  bg_on_data=False, 
                                          alpha=0.8,
                                  output_file=plot_fn,
                                  darkness=0.8, 
                                  threshold=1, title=f'Camcan {t} {tp}',
                                 )
        plt.close()
        print(plot_fn)

found 117 for rest optt
(117, 86741)
CamCAN/plots/group_average_rest_tphate_optt_whole_brain_SL_rad5.png
found 117 for rest autocorr
(117, 86741)
CamCAN/plots/group_average_rest_tphate_autocorr_whole_brain_SL_rad5.png
found 117 for movie optt
(117, 86741)
CamCAN/plots/group_average_movie_tphate_optt_whole_brain_SL_rad5.png
found 117 for movie autocorr
(117, 86741)
CamCAN/plots/group_average_movie_tphate_autocorr_whole_brain_SL_rad5.png
found 117 for SMT optt
(117, 86741)
CamCAN/plots/group_average_SMT_tphate_optt_whole_brain_SL_rad5.png
found 117 for SMT autocorr
(117, 86741)
CamCAN/plots/group_average_SMT_tphate_autocorr_whole_brain_SL_rad5.png


In [23]:
# reorganize data

outdir = '/gpfs/milgram/project/casey/elb77/CamCAN/fMRI_organized/'
all_subjects = utils.get_intersecting_subjects(subject_filter=0)

for S in all_subjects:
    s = S.replace('sub-', '')
    dirname = f'{BASE_DIR_CAMCAN}/{CAMCAN_PREPROC_MIDSTR}/{s}/'
    old_fn = glob.glob(os.path.join(dirname, 'Movie', f'*fMR*_{s}*.nii'))[0]
    new_fn = f'{outdir}/movie/{S}_MOVIE.nii'
    shutil.copy(old_fn, new_fn)
    
    old_fn = glob.glob(os.path.join(dirname, 'SMT', f'*fMR*_{s}*.nii'))[0]
    new_fn = f'{outdir}/smt/{S}_SMT.nii'
    shutil.copy(old_fn, new_fn)
    
    old_fn = glob.glob(os.path.join(dirname, 'Rest', f'*fMR*_{s}*.nii'))[0]
    new_fn = f'{outdir}/rest/{S}_REST.nii'
    shutil.copy(old_fn, new_fn)
    print(S)
    
    


sub-CC110033
sub-CC110037
sub-CC110045
sub-CC110056
sub-CC110069
sub-CC110087
sub-CC110098
sub-CC110101
sub-CC110126
sub-CC110174
sub-CC110182
sub-CC110187
sub-CC110319
sub-CC110411
sub-CC110606
sub-CC112141
sub-CC120008
sub-CC120049
sub-CC120061
sub-CC120065
sub-CC120120
sub-CC120123
sub-CC120166
sub-CC120182
sub-CC120208
sub-CC120218
sub-CC120234
sub-CC120264
sub-CC120276
sub-CC120286
sub-CC120309
sub-CC120313
sub-CC120319
sub-CC120347
sub-CC120376
sub-CC120409
sub-CC120462
sub-CC120469
sub-CC120470
sub-CC120550
sub-CC120640
sub-CC120727
sub-CC120764
sub-CC120795
sub-CC120816
sub-CC120987
sub-CC121106
sub-CC121111
sub-CC121144
sub-CC121158
sub-CC121194
sub-CC121200
sub-CC121317
sub-CC121397
sub-CC121411
sub-CC121428
sub-CC121479
sub-CC121685
sub-CC121795
sub-CC122172
sub-CC122405
sub-CC122620
sub-CC210023
sub-CC210051
sub-CC210088
sub-CC210124
sub-CC210148
sub-CC210172
sub-CC210182
sub-CC210250
sub-CC210304
sub-CC210314
sub-CC210422
sub-CC210519
sub-CC210526
sub-CC210617
sub-CC210657

In [17]:
df_movie = pd.read_csv(CAMCAN_PARTICIPANT_FILES['MOVIE'], sep='\t')
print(df_movie.shape, len(df_movie.participant_id.unique()))
df_movie[df_movie['age']<20]

(649, 6) 649


,participant_id,age,hand,gender_text,gender_code,tiv_cubicmm
1,sub-CC110037,18,89.0,MALE,1,1391950
10,sub-CC110182,18,80.0,FEMALE,2,1323662
18,sub-CC120061,19,70.0,MALE,1,1598322
21,sub-CC120123,19,56.0,FEMALE,2,1260867
34,sub-CC120376,18,79.0,FEMALE,2,1236457
35,sub-CC120409,18,100.0,MALE,1,1624266
36,sub-CC120462,18,-80.0,FEMALE,2,1327319
39,sub-CC120550,19,70.0,MALE,1,1471032
47,sub-CC121111,18,90.0,MALE,1,1696915


In [18]:
df_rest = pd.read_csv(CAMCAN_PARTICIPANT_FILES['REST'], sep='\t')
print(df_rest.shape, len(df_rest.participant_id.unique()))
df_rest[df_rest['age']<20]

(652, 6) 652


,participant_id,age,hand,gender_text,gender_code,tiv_cubicmm
1,sub-CC110037,18,89.0,MALE,1,1391950
10,sub-CC110182,18,80.0,FEMALE,2,1323662
18,sub-CC120061,19,70.0,MALE,1,1598322
21,sub-CC120123,19,56.0,FEMALE,2,1260867
34,sub-CC120376,18,79.0,FEMALE,2,1236457
35,sub-CC120409,18,100.0,MALE,1,1624266
36,sub-CC120462,18,-80.0,FEMALE,2,1327319
39,sub-CC120550,19,70.0,MALE,1,1471032
47,sub-CC121111,18,90.0,MALE,1,1696915


In [19]:
df_smt = pd.read_csv(CAMCAN_PARTICIPANT_FILES['SMT'], sep='\t')
print(df_smt.shape, len(df_smt.participant_id.unique()))
df_smt[df_smt['age']<20]

(651, 6) 651


,participant_id,age,hand,gender_text,gender_code,tiv_cubicmm
1,sub-CC110037,18,89.0,MALE,1,1391950
10,sub-CC110182,18,80.0,FEMALE,2,1323662
18,sub-CC120061,19,70.0,MALE,1,1598322
21,sub-CC120123,19,56.0,FEMALE,2,1260867
34,sub-CC120376,18,79.0,FEMALE,2,1236457
35,sub-CC120409,18,100.0,MALE,1,1624266
36,sub-CC120462,18,-80.0,FEMALE,2,1327319
39,sub-CC120550,19,70.0,MALE,1,1471032
47,sub-CC121111,18,90.0,MALE,1,1696915
